In [ ]:
import torch

PYG_URL = 'https://data.pyg.org/whl/torch-2.11+cpu.html' # Using the PYG_URL from the kernel state
!pip install torch_geometric -f {PYG_URL}

import pandas as pd
import numpy as np

from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split

from torch_geometric.data import Data
from torch_geometric.utils import to_undirected

Looking in links: https://data.pyg.org/whl/torch-2.11+cpu.html


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv(
    "/content/drive/MyDrive/Real_school_data_with_proxy.csv"
)

print(df.shape)
df.head()

(1000, 46)


,Student ID,School,Gender,Class level,Parental educational level,Week1_attendance,Week2_attendance,Week3_attendance,Week4_attendance,Week5_attendance,...,level_prefix,long_commute_flag,long_walk_flag,academic_domain,attendance_domain,socioeconomic_domain,engagement_domain,accessibility_domain,risk_score,dropout_risk
0,1,Shining Star Preparatory,Female,4A,SHS,1.0,1.00,0.40,1.0,1.0,...,JHS,0,0,0.196491,0.313559,0.375,0.333333,0.50,0.311348,0
1,2,Shining Star Preparatory,Male,4B,SHS,1.0,1.00,0.25,1.0,1.0,...,JHS,0,0,0.328313,0.127119,0.625,0.333333,0.50,0.344963,1
2,3,Shining Star Preparatory,Male,7B3,Tertiary,0.5,0.75,1.00,1.0,1.0,...,JHS,0,0,0.334251,0.432203,0.125,0.333333,0.50,0.338270,1
3,4,Weweso MA,Male,5,JHS,0.6,1.00,1.00,1.0,1.0,...,JHS,0,0,0.707800,0.288136,0.500,0.333333,0.25,0.457114,2
4,5,Weweso MA,Male,6B,Tertiary,0.0,1.00,1.00,1.0,1.0,...,JHS,0,0,0.270229,0.169492,0.375,0.166667,0.25,0.248583,0


In [ ]:
print(df.columns.tolist())

['Student ID', 'School', 'Gender', 'Class level', 'Parental educational level', 'Week1_attendance', 'Week2_attendance', 'Week3_attendance', 'Week4_attendance', 'Week5_attendance', 'Week6_attendance', 'Week7_attendance', 'Week8_attendance', 'Week9_attendance', 'Week10_attendance', 'Week11_attendance', 'Week12_attendance', 'Week13_attendance', 'Week14_attendance', 'Semester 1 average', 'Semester 2 average', 'Semester difference', 'Household income level', 'Family dropout history', 'Child labor involvement', 'Travel time to school', 'Mode of transport', 'Teacher relationship quality', 'Peer relationship quality', 'Extra-curricular activities', 'Section', 'Household income level (standardized)', 'Travel time to school (minutes)', 'grade_number', 'stream_letter', 'subgroup', 'level_prefix', 'long_commute_flag', 'long_walk_flag', 'academic_domain', 'attendance_domain', 'socioeconomic_domain', 'engagement_domain', 'accessibility_domain', 'risk_score', 'dropout_risk']


In [ ]:
target_column = "dropout_risk"

X = df.drop(columns=[
    "Student ID",
    "risk_score",
    target_column
])

y = df[target_column]

In [ ]:
print("Feature matrix:", X.shape)
print("Labels:", y.shape)

Feature matrix: (1000, 43)
Labels: (1000,)


In [ ]:
X = pd.get_dummies(X, drop_first=True) # One-hot encode categorical features
X = X.values.astype(np.float32)

y = y.values.astype(np.int64)

X_tensor = torch.tensor(X)

y_tensor = torch.tensor(y)

print(X_tensor.shape)
print(y_tensor.shape)

torch.Size([1000, 152])
torch.Size([1000])


In [ ]:
print("Mean of first 10 features:")
print(X_tensor.mean(dim=0)[:10])

print("\nStandard deviation of first 10 features:")
print(X_tensor.std(dim=0)[:10])

Mean of first 10 features:
tensor([0.7933, 0.8805, 0.8571, 0.8661, 0.8926, 0.9269, 0.8905, 0.9293, 0.8985,
        0.9482])

Standard deviation of first 10 features:
tensor([0.3306, 0.2759, 0.2661, 0.3172, 0.2576, 0.2342, 0.2897, 0.1989, 0.2549,
        0.1637])


In [31]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = X_scaled.astype(np.float32)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

print("Scaled feature matrix:", X_tensor.shape)

print("\nMean of first 10 features:")
print(X_tensor.mean(dim=0)[:10])

print("\nStandard deviation of first 10 features:")
print(X_tensor.std(dim=0)[:10])

Scaled feature matrix: torch.Size([1000, 152])

Mean of first 10 features:
tensor([ 4.7684e-09,  5.7220e-09,  1.9073e-09,  1.3351e-08, -2.8610e-08,
         3.8147e-09,  1.7881e-08,  9.5367e-10, -1.5020e-08,  1.1921e-08])

Standard deviation of first 10 features:
tensor([1.0005, 1.0005, 1.0005, 1.0005, 1.0005, 1.0005, 1.0005, 1.0005, 1.0005,
        1.0005])


In [32]:
from sklearn.neighbors import NearestNeighbors

k = 5

knn = NearestNeighbors(
    n_neighbors=k + 1,
    metric="cosine",
    algorithm="brute"
)

knn.fit(X_scaled)

distances, indices = knn.kneighbors(X_scaled)

print("Distance matrix shape:", distances.shape)
print("Neighbour index matrix shape:", indices.shape)

Distance matrix shape: (1000, 6)
Neighbour index matrix shape: (1000, 6)


In [33]:
# Remove the first neighbour (the student itself)
knn_distances = distances[:, 1:]
knn_indices = indices[:, 1:]

print("KNN distances shape:", knn_distances.shape)
print("KNN indices shape:", knn_indices.shape)

KNN distances shape: (1000, 5)
KNN indices shape: (1000, 5)


In [34]:
for student in range(5):
    print(f"Student {student}:")
    print("  Neighbours:", knn_indices[student])
    print("  Distances: ", knn_distances[student])
    print()

Student 0:
  Neighbours: [818 461 352  36 903]
  Distances:  [0.21464366 0.22605807 0.2305938  0.73345506 0.7582952 ]

Student 1:
  Neighbours: [807 817 624 762 250]
  Distances:  [0.2638693  0.2939477  0.32393432 0.32405025 0.34318703]

Student 2:
  Neighbours: [780 965 633 323 228]
  Distances:  [0.42112064 0.4456179  0.53090644 0.53493345 0.5463175 ]

Student 3:
  Neighbours: [918 146 871 950 552]
  Distances:  [0.3111909  0.4000451  0.43301618 0.4448529  0.46436793]

Student 4:
  Neighbours: [ 20 858 357 179 463]
  Distances:  [0.46606094 0.48268944 0.48269117 0.4929781  0.49684906]



In [35]:
# Convert cosine distances to cosine similarities
knn_similarity = 1 - knn_distances

print("Similarity shape:", knn_similarity.shape)

for student in range(5):
    print(f"Student {student}:")
    print("  Neighbours:", knn_indices[student])
    print("  Similarities:", knn_similarity[student])
    print()

Similarity shape: (1000, 5)
Student 0:
  Neighbours: [818 461 352  36 903]
  Similarities: [0.78535634 0.77394193 0.7694062  0.26654494 0.24170482]

Student 1:
  Neighbours: [807 817 624 762 250]
  Similarities: [0.7361307  0.7060523  0.6760657  0.67594975 0.65681297]

Student 2:
  Neighbours: [780 965 633 323 228]
  Similarities: [0.57887936 0.5543821  0.46909356 0.46506655 0.45368248]

Student 3:
  Neighbours: [918 146 871 950 552]
  Similarities: [0.6888091 0.5999549 0.5669838 0.5551471 0.5356321]

Student 4:
  Neighbours: [ 20 858 357 179 463]
  Similarities: [0.53393906 0.51731056 0.51730883 0.5070219  0.50315094]



In [36]:
# Number of students
num_nodes = X_tensor.shape[0]

# Create source nodes
source_nodes = np.repeat(
    np.arange(num_nodes),
    k
)

# Create destination nodes
target_nodes = knn_indices.reshape(-1)

# Create edge weights
edge_weights = knn_similarity.reshape(-1)

# Convert to PyTorch tensors
edge_index = torch.tensor(
    np.vstack([source_nodes, target_nodes]),
    dtype=torch.long
)

edge_weight = torch.tensor(
    edge_weights,
    dtype=torch.float32
)

print("edge_index shape:", edge_index.shape)
print("edge_weight shape:", edge_weight.shape)

edge_index shape: torch.Size([2, 5000])
edge_weight shape: torch.Size([5000])


In [37]:
# Create all directed + reverse edges
src = np.concatenate([
    source_nodes,
    target_nodes
])

dst = np.concatenate([
    target_nodes,
    source_nodes
])

weights = np.concatenate([
    edge_weights, # Changed from directed_weights
    edge_weights  # Changed from directed_weights
])

# Build a pandas DataFrame to handle duplicate edges
edge_df = pd.DataFrame({
    "source": src,
    "target": dst,
    "weight": weights
})

# For duplicate pairs, keep the maximum similarity
edge_df = (
    edge_df
    .groupby(["source", "target"], as_index=False)["weight"]
    .max()
)

# Convert to PyTorch tensors
edge_index = torch.tensor(
    edge_df[["source", "target"]].values.T,
    dtype=torch.long
)

edge_weight = torch.tensor(
    edge_df["weight"].values,
    dtype=torch.float32
)

print("Corrected edge_index shape:", edge_index.shape)
print("Corrected edge_weight shape:", edge_weight.shape)

Corrected edge_index shape: torch.Size([2, 6580])
Corrected edge_weight shape: torch.Size([6580])


In [38]:
from torch_geometric.utils import contains_self_loops

print("Number of nodes:", num_nodes)
print("Number of edges:", edge_index.shape[1])

print("Contains self-loops:",
      contains_self_loops(edge_index))

print("\nEdge-weight statistics:")
print("Minimum:", edge_weight.min().item())
print("Maximum:", edge_weight.max().item())
print("Mean:", edge_weight.mean().item())
print("Median:", edge_weight.median().item())

Number of nodes: 1000
Number of edges: 6580
Contains self-loops: False

Edge-weight statistics:
Minimum: 0.06654709577560425
Maximum: 0.9592154026031494
Mean: 0.5033146739006042
Median: 0.5061290860176086


In [39]:
from torch_geometric.utils import degree

node_degree = degree(
    edge_index[0],
    num_nodes=num_nodes
)

print("Minimum degree:", node_degree.min().item())
print("Maximum degree:", node_degree.max().item())
print("Average degree:", node_degree.mean().item())

print(
    "Number of isolated nodes:",
    (node_degree == 0).sum().item()
)

Minimum degree: 5.0
Maximum degree: 13.0
Average degree: 6.579999923706055
Number of isolated nodes: 0


In [42]:
from sklearn.model_selection import train_test_split

# Convert labels to NumPy
y_np = y_tensor.numpy()

# All node indices
node_indices = np.arange(num_nodes)

# 70% training, 30% temporary
train_idx, temp_idx = train_test_split(
    node_indices,
    test_size=0.30,
    stratify=y_np,
    random_state=42
)

# Split the remaining 30% equally:
# 15% validation, 15% test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=y_np[temp_idx],
    random_state=42
)

print("Training indices:", len(train_idx))
print("Validation indices:", len(val_idx))
print("Testing indices:", len(test_idx))

Training indices: 700
Validation indices: 150
Testing indices: 150


In [43]:
train_mask = torch.zeros(
    num_nodes,
    dtype=torch.bool
)

val_mask = torch.zeros(
    num_nodes,
    dtype=torch.bool
)

test_mask = torch.zeros(
    num_nodes,
    dtype=torch.bool
)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

print("Training mask:", train_mask.sum().item())
print("Validation mask:", val_mask.sum().item())
print("Test mask:", test_mask.sum().item())

Training mask: 700
Validation mask: 150
Test mask: 150


In [44]:
train_mask = torch.zeros(
    num_nodes,
    dtype=torch.bool
)

val_mask = torch.zeros(
    num_nodes,
    dtype=torch.bool
)

test_mask = torch.zeros(
    num_nodes,
    dtype=torch.bool
)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

print("Training mask:", train_mask.sum().item())
print("Validation mask:", val_mask.sum().item())
print("Test mask:", test_mask.sum().item())

Training mask: 700
Validation mask: 150
Test mask: 150


In [45]:
print(
    "Train ∩ Validation:",
    (train_mask & val_mask).sum().item()
)

print(
    "Train ∩ Test:",
    (train_mask & test_mask).sum().item()
)

print(
    "Validation ∩ Test:",
    (val_mask & test_mask).sum().item()
)

print(
    "Total assigned nodes:",
    (
        train_mask.sum()
        + val_mask.sum()
        + test_mask.sum()
    ).item()
)

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
Total assigned nodes: 1000


In [46]:
from torch_geometric.data import Data

data = Data(
    x=X_tensor,
    edge_index=edge_index,
    edge_weight=edge_weight,
    y=y_tensor,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask
)

print(data)

Data(x=[1000, 152], edge_index=[2, 6580], y=[1000], edge_weight=[6580], train_mask=[1000], val_mask=[1000], test_mask=[1000])


In [47]:
from torch_geometric.data import Data

data = Data(
    x=X_tensor,
    edge_index=edge_index,
    edge_weight=edge_weight,
    y=y_tensor,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask
)

print(data)

Data(x=[1000, 152], edge_index=[2, 6580], y=[1000], edge_weight=[6580], train_mask=[1000], val_mask=[1000], test_mask=[1000])


In [48]:
print("Number of nodes:", data.num_nodes)
print("Number of features:", data.num_node_features)
print("Number of edges:", data.num_edges)

print("\nX shape:", data.x.shape)
print("y shape:", data.y.shape)
print("edge_index shape:", data.edge_index.shape)
print("edge_weight shape:", data.edge_weight.shape)

print("\nTrain nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())

Number of nodes: 1000
Number of features: 152
Number of edges: 6580

X shape: torch.Size([1000, 152])
y shape: torch.Size([1000])
edge_index shape: torch.Size([2, 6580])
edge_weight shape: torch.Size([6580])

Train nodes: 700
Validation nodes: 150
Test nodes: 150


In [49]:
print("========== FINAL GRAPH CHECK ==========")

print("Nodes:", data.num_nodes)
print("Features per node:", data.num_node_features)
print("Edges:", data.num_edges)

print("\nEdge weights:")
print(" Min:", data.edge_weight.min().item())
print(" Max:", data.edge_weight.max().item())
print(" Mean:", data.edge_weight.mean().item())

print("\nMasks:")
print(" Train:", data.train_mask.sum().item())
print(" Validation:", data.val_mask.sum().item())
print(" Test:", data.test_mask.sum().item())

print("\nGraph valid:",
      data.validate(raise_on_error=False))

========== FINAL GRAPH CHECK ==========
Nodes: 1000
Features per node: 152
Edges: 6580

Edge weights:
 Min: 0.06654709577560425
 Max: 0.9592154026031494
 Mean: 0.5033146739006042

Masks:
 Train: 700
 Validation: 150
 Test: 150

Graph valid: True


In [50]:
torch.save(
    data,
    "/content/drive/MyDrive/student_dropout_cosine_knn_graph.pt"
)

print("Graph saved successfully.")

Graph saved successfully.
